This is my model One demostartion :  BM25 from the WANDS TSV file , we have three datasets namley 
1)Product df , which is the product catalogue of what the store has and what can be search for by customer.42994 products 
2)The second dataset is the Query df of what customers have actually searched for 480 queries 
And than the final dataset is the 3)Label df of human relevance search results and the key evaluation that our model will be evaluated on 233 448 human labeled query and product relevance pairs with the label being , exact , partial or irrelevant match.

Data preprocessing techniques  BM25 a non supervised  lexical retrieval model ( how may words overlap the query and the product.),  BM25 scores and ranks documents according to their relevance to a query based on lexical metching . I am testing this model as a information retrieval and ranking task and towards the end of the notebook I expirement with finding products and seeing how they rank and using the label df as guide.  




In [19]:
import pandas as pd
import numpy as np
import re


query_df = pd.read_csv('query.csv', sep='\t')
product_df = pd.read_csv('product.csv.zip', sep='\t')
label_df = pd.read_csv('label.csv.zip', sep='\t')


print(f"Query shape: {query_df.shape}")
print(f"Products shape: {product_df.shape}")
print(f"Labels shape: {label_df.shape}")

Query shape: (480, 3)
Products shape: (42994, 9)
Labels shape: (233448, 4)


In [20]:

dfs = {'Query': query_df, 'Product': product_df, 'Label': label_df}

for name, df in dfs.items():
    print(f"\nMissing values in {name}:")
    print(df.isna().sum())


Missing values in Query:
query_id       0
query          0
query_class    6
dtype: int64

Missing values in Product:
product_id                0
product_name              0
product_class          2852
category hierarchy     1556
product_description    6008
product_features          0
rating_count           9452
average_rating         9452
review_count           9452
dtype: int64

Missing values in Label:
id            0
query_id      0
product_id    0
label         0
dtype: int64


In [21]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()


cols_to_clean = ['product_name', 'product_class', 'category hierarchy', 'product_description', 'product_features']

for col in cols_to_clean:
    product_df[col] = product_df[col].fillna('')
    product_df[col + '_clean'] = product_df[col].apply(clean_text)

Now as part of the BM25 model , we are compile and combine the colums from the products datatable . 

In [22]:
cols_to_clean = ['product_name', 'product_class', 'category hierarchy', 'product_description', 'product_features']

for col in cols_to_clean:
    product_df[col] = product_df[col].fillna('')
    product_df[col + '_clean'] = product_df[col].apply(clean_text)

In [23]:
product_df['search_document'] = (
    product_df['product_name_clean'] + " " + 
    product_df['product_class_clean'] + " " + 
    product_df['category hierarchy_clean'] + " " + 
    product_df['product_description_clean'] + " " + 
    product_df['product_features_clean']
)

In [7]:
product_df[['product_name', 'search_document']].head()

,product_name,search_document
0,solid wood platform bed,solid wood platform bed beds furniture bedroo...
1,all-clad 7 qt . slow cooker,allclad 7 qt slow cooker slow cookers kitchen...
2,all-clad electrics 6.5 qt . slow cooker,allclad electrics 65 qt slow cooker slow cook...
3,all-clad all professional tools pizza cutter,allclad all professional tools pizza cutter sl...
4,baldwin prestige alcott passage knob with roun...,baldwin prestige alcott passage knob with roun...


In [8]:
from rank_bm25 import BM25Okapi

# Tokenize documents for BM25 indexing
tokenized_corpus = [doc.split() for doc in product_df['search_document']]

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

Here is where i have installed and created the retrieval model as to create df into product search document 
All product info from the product data set have now been made one document search document and this info will be in one place. Than later compared to the search query dataframe and the label df. 

In [24]:
print("Sample tokenized document:")
print(tokenized_corpus[0])

Sample tokenized document:
['solid', 'wood', 'platform', 'bed', 'beds', 'furniture', 'bedroom', 'furniture', 'beds', 'headboards', 'beds', 'twin', 'beds', 'good', 'deep', 'sleep', 'can', 'be', 'quite', 'difficult', 'to', 'have', 'in', 'this', 'busy', 'age', 'fortunately', 'there', 's', 'an', 'antidote', 'to', 'such', 'a', 'problem', 'a', 'nice', 'quality', 'bed', 'frame', 'like', 'the', 'acacia', 'kaylin', 'solidly', 'constructed', 'from', 'acacia', 'wood', 'this', 'bed', 'frame', 'will', 'stand', 'the', 'test', 'of', 'time', 'and', 'is', 'fit', 'to', 'rest', 'your', 'shoulders', 'on', 'for', 'years', 'and', 'years', 'its', 'sleek', 'natural', 'wood', 'grain', 'appearance', 'provides', 'a', 'pleasant', 'aesthetic', 'to', 'adorn', 'any', 'bedroom', 'acting', 'both', 'as', 'a', 'decorative', 'piece', 'as', 'well', 'as', 'a', 'place', 'to', 'give', 'comfort', 'after', 'a', 'hard', 'day', 'of', 'work', 'our', 'bed', 'frame', 'is', 'designed', 'to', 'give', 'ample', 'underbed', 'space', 'fo

In [25]:
bm25 = BM25Okapi(tokenized_corpus)

At this point I tested one search query which was salon chair , to test what the model would return 

In [26]:
test_query = "salon chair"
tokenized_query = clean_text(test_query).split()


top_docs = bm25.get_top_n(tokenized_query, product_df['search_document'].tolist(), n=5)

for i, doc in enumerate(top_docs, 1):
    print(f"{i}. {doc}")

1. massage chair massage chairs furniture  living room furniture  chairs  seating  massage chairs features heavy duty steel frame  premium chrome round base  heavy duty hydraulic pump make height adjustment easier  doublereinforced saddle sticking prevents ripping and tearing  small cell high density foam for extra comfort of your client  easy to assemble and clean  barber chair salon chair styling chair salon chairs hair styling chair haircut chair salon chair haircut chair salon barber chair heavy duty barber chair shampoo chair hydraulic salon chair salon stool chair beauty hair chair shampoo chair adjustable swivel barber chairs shampoo chair with leg rest salon stool for hair stylist stool salon haircut chair home hydraulic lift styling chair heavy duty salon chair overallheighttoptobottom355productcare  dry clothupholsterymaterial  faux leathercolor  blackseatdepthfronttoback195seatwidthsidetoside195recliningtype  manualoveralldepthfronttoback23removablecushionlocation  backweigh

The BM25 than returned search of relevent product infomation however due to the output having long product descriptions the rank was difficult to intrepret so i had to create a table o tha relevent fields are displayed 

In [27]:
test_query = "salon chair"
tokenized_query = clean_text(test_query).split()
bm25_scores = bm25.get_scores(tokenized_query)


top_10_idx = np.argsort(bm25_scores)[-10:][::-1]


cols = ['product_id', 'product_name', 'product_class', 'category hierarchy']
bm25_results = product_df.iloc[top_10_idx][cols].copy()

bm25_results['bm25_score'] = bm25_scores[top_10_idx]
bm25_results.insert(0, 'rank', range(1, 11))

bm25_results.reset_index(drop=True)

,rank,product_id,product_name,product_class,category hierarchy,bm25_score
0,1,15612,massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,16.914649
1,2,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,15.212085
2,3,7466,reclining massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.128911
3,4,7467,reclining faux leather massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.090445
4,5,7465,hair salon chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.089227
5,6,22462,hard floor straight round chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.543773
6,7,22463,hard floor straight rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.340485
7,8,25431,barberpub salon massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,13.199099
8,9,36889,beveled rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats,12.946083
9,10,29744,genuine leather reclining adjustable width mas...,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,12.831897


In this phase I was testing the BM25 model with the salon chair example , before evlauating the full model. This example illustates how the BM25 scores and ranks products 

In [28]:
query_id = query_df.loc[query_df['query'] == 'salon chair', 'query_id'].iloc[0]
ground_truth = label_df[label_df['query_id'] == query_id][['product_id', 'label']]


bm25_evaluation = bm25_results.merge(ground_truth, on='product_id', how='left')
bm25_evaluation['label'] = bm25_evaluation['label'].fillna('Unjudged')


print(f"Query ID: {query_id}")
bm25_evaluation

Query ID: 0


,rank,product_id,product_name,product_class,category hierarchy,bm25_score,label
0,1,15612,massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,16.914649,Exact
1,2,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,15.212085,Exact
2,3,7466,reclining massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.128911,Exact
3,4,7467,reclining faux leather massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.090445,Exact
4,5,7465,hair salon chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.089227,Exact
5,6,22462,hard floor straight round chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.543773,Irrelevant
6,7,22463,hard floor straight rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.340485,Irrelevant
7,8,25431,barberpub salon massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,13.199099,Exact
8,9,36889,beveled rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats,12.946083,Irrelevant
9,10,29744,genuine leather reclining adjustable width mas...,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,12.831897,Exact


Now I am evaluating the Models ranking and the human label judgment aspect . At this level all three datasets have been combined ( product df , query of customer and the judgment label) with the score of the model for the top 10 products , now I will be evaluating the metrics of the BM25 model for these 10 products.

In [30]:
import numpy as np


label_map = {'Exact': 2, 'Partial': 1, 'Irrelevant': 0, 'Unjudged': 0}
relevances = bm25_evaluation['label'].map(label_map).fillna(0).values

# 1. Precision@10
relevant_retrieved = np.sum(relevances > 0)
precision_at_10 = relevant_retrieved / len(relevances)

# 2. Recall@10 (against all known relevant items in ground truth)
total_relevant_in_gt = len(ground_truth[ground_truth['label'].isin(['Exact', 'Partial'])])
recall_at_10 = relevant_retrieved / total_relevant_in_gt if total_relevant_in_gt > 0 else 0.0

# 3. NDCG@10 Calculation
def compute_dcg(rel):
    discounts = np.log2(np.arange(len(rel)) + 2)
    return np.sum((2**rel - 1) / discounts)

dcg = compute_dcg(relevances)
idcg = compute_dcg(np.sort(relevances)[::-1])
ndcg_at_10 = dcg / idcg if idcg > 0 else 0.0


print(f"Precision@10 : {precision_at_10:.4f}")
print(f"Recall@10    : {recall_at_10:.4f}")
print(f"NDCG@10      : {ndcg_at_10:.4f}")

Precision@10 : 0.7000
Recall@10    : 0.0023
NDCG@10      : 0.9766


Above are the evaluation scores of the model of 10 products , however I wanted to futher investigate indivaual query to observe where the model succeeds and fails.

because the intial evaluation focused on the serch query of salon chair as an demonstrartion of how the model performs on the evlauaton metrics , I than had to repeat the and investivate the performance on diffrent customer searches and understand what were the best and worst query results compared the model results and the label results 

In [31]:
def evaluate_query(search_query):

    # Find the query ID
    query_id = query_df.loc[
        query_df['query'] == search_query, 'query_id'
    ].iloc[0]


    ground_truth = label_df[
        label_df['query_id'] == query_id
    ][['product_id', 'label']]

    # BM25 search
    tokenized_query = clean_text(search_query).split()
    scores = bm25.get_scores(tokenized_query)

 
    top_10_idx = np.argsort(scores)[-10:][::-1]

    results = product_df.iloc[top_10_idx][
        ['product_id', 'product_name']
    ].copy()

    results['bm25_score'] = scores[top_10_idx]
    results['rank'] = range(1, 11)


    results = results.merge(
        ground_truth,
        on='product_id',
        how='left'
    )

    results['label'] = results['label'].fillna('Unjudged')

    return results

I than wanted to investage why a query like boho bed frame didnt do so well based on the evaluation functions , this table illustartes the rank of the BM25 model and the human label , this is important because we can identify where the model ranks relevent products poorly. 

In [17]:
evaluate_query("boho bed frame")

,product_id,product_name,bm25_score,rank,label
0,39844,dorinda king solid wood platform bed,11.648653,1,Unjudged
1,4350,rotz upholstered low profile platform bed,11.321974,2,Partial
2,13495,catherine mcdonald boho island comforter,10.974378,3,Unjudged
3,28507,tortelli never forget teak picture frame,10.169041,4,Unjudged
4,26515,tacoma twin platform bed,9.521151,5,Unjudged
5,3555,brea open-frame headboard,9.297063,6,Irrelevant
6,25405,natural fibers cachepot,8.844331,7,Unjudged
7,25404,carice natural fibers cachepot,8.805112,8,Unjudged
8,41477,engstrom 40 '' bed frame,8.648699,9,Irrelevant
9,30986,schurman abstract tufted wool/cotton beige are...,8.632847,10,Unjudged


To the evaluation i than applied the boho bed frame query to the model. the output was a table that displays 10 results with a mixed relevence judgemet , that labeled partial , irrelevent and unjudged . This is strange and I had to investgate if there are products that are are labeled as relevent but BM25 missed out of the top 10 searches , therefore i had to investigate the human labelled relevent products for this query 

In [32]:

test_query = "boho bed frame"


query_id = query_df.loc[
    query_df['query'] == test_query, 'query_id'
].iloc[0]


exact_products = label_df[
    (label_df['query_id'] == query_id) &
    (label_df['label'] == 'Exact')
]


tokenized_query = clean_text(test_query).split()
bm25_scores = bm25.get_scores(tokenized_query)

# Rank all products
ranked_idx = np.argsort(bm25_scores)[::-1]

# Create a ranking table
ranked_products = product_df.iloc[ranked_idx][
    ['product_id', 'product_name']
].copy()

ranked_products['BM25_rank'] = range(1, len(ranked_products) + 1)

# Match the human Exact products with their BM25 rank
exact_results = exact_products.merge(
    ranked_products,
    on='product_id',
    how='left'
)

# Show the result
exact_results[
    ['product_id', 'product_name', 'label', 'BM25_rank']
]

,product_id,product_name,label,BM25_rank
0,27655,boho chic platform bed,Exact,26


In [33]:
import numpy as np
from sklearn.metrics import ndcg_score

precision_scores = []
recall_scores = []
ndcg_scores = []

for _, row in query_df.iterrows():

    # Get query and BM25 scores
    query_tokens = row['query'].lower().split()
    scores = bm25.get_scores(query_tokens)

    # Rank products and select Top 10
    top_10 = np.argsort(scores)[::-1][:10]

    top_products = product_df.iloc[top_10][['product_id']].copy()
    top_products['query_id'] = row['query_id']

    # Add human relevance labels
    results = top_products.merge(
        label_df[['query_id', 'product_id', 'label']],
        on=['query_id', 'product_id'],
        how='left'
    )

    # ----- PRECISION@10 -----
    judged = results.dropna(subset=['label'])

    if len(judged) > 0:
        relevant = judged['label'].isin(['Exact', 'Partial']).sum()
        precision_scores.append(relevant / len(judged))

    # ----- RECALL@10 -----
    all_relevant = label_df[
        (label_df['query_id'] == row['query_id']) &
        (label_df['label'].isin(['Exact', 'Partial']))
    ]

    retrieved_relevant = results['label'].isin(['Exact', 'Partial']).sum()

    if len(all_relevant) > 0:
        recall_scores.append(retrieved_relevant / len(all_relevant))

    # ----- NDCG@10 -----
    relevance = results['label'].map({
        'Exact': 2,
        'Partial': 1,
        'Irrelevant': 0
    }).fillna(0).to_numpy()

    if relevance.sum() > 0:
        ideal = np.sort(relevance)[::-1]

        ndcg_scores.append(
            ndcg_score([ideal], [relevance], k=10)
        )


# FINAL RESULTS
bm25_precision = np.mean(precision_scores)
bm25_recall = np.mean(recall_scores)
bm25_ndcg = np.mean(ndcg_scores)

print("BM25 Judged Precision@10:", round(bm25_precision * 100, 2), "%")
print("BM25 Recall@10:", round(bm25_recall * 100, 2), "%")
print("BM25 NDCG@10:", round(bm25_ndcg * 100, 2), "%")

BM25 Judged Precision@10: 91.49 %
BM25 Recall@10: 5.6 %
BM25 NDCG@10: 88.45 %


this output demostartes a limitation of the model , in that boho chic platfrom bed was an exact match to the search query of boho bed frame according to the human label , the model is able to retive products but they are still ranked lower based on the words the customers used in thier query being diffrent to the the product infomation . 